# 42 Submission 13

Final Kaggle submission based on the Experiment 41D winning rank blend.

In [1]:

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from catboost import CatBoostClassifier


train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

TARGET = 'Will_Buy_EV'
ID_COL = 'id'

y = (
    train[TARGET]
    .astype(str)
    .str.strip()
    .map({'No': 0, 'Yes': 1})
    .astype(int)
)

X_train_raw = train.drop(columns=[TARGET, ID_COL]).copy()
X_test_raw = test.drop(columns=[ID_COL]).copy()

numeric_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level'
]

categorical_cols = [
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
]


# ============================================================
# EXACT 33B IDENTITY ENCODING
# ============================================================

def identity_key(series):
    return series.astype('string').fillna('__MISSING__')


def make_smoothed_mapping(keys, target, smoothing=20):
    temp = pd.DataFrame({
        'key': identity_key(keys),
        'target': target.values
    })

    stats = temp.groupby('key')['target'].agg(['count', 'mean'])
    global_mean = target.mean()

    mapping = (
        stats['count'] * stats['mean']
        + smoothing * global_mean
    ) / (stats['count'] + smoothing)

    return mapping, global_mean


def add_full_identity_features(
    X_fit,
    X_apply,
    target,
    columns,
    prefix,
    smoothing=20
):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    global_mean = target.mean()

    for col in columns:
        fit_keys = identity_key(X_fit[col])
        apply_keys = identity_key(X_apply[col])

        mapping, _ = make_smoothed_mapping(
            fit_keys,
            target,
            smoothing
        )

        X_fit[f'{prefix}_{col}_target'] = (
            fit_keys
            .map(mapping)
            .fillna(global_mean)
            .astype(float)
        )

        X_apply[f'{prefix}_{col}_target'] = (
            apply_keys
            .map(mapping)
            .fillna(global_mean)
            .astype(float)
        )

        freq = fit_keys.value_counts(normalize=True)

        X_fit[f'{prefix}_{col}_freq'] = (
            fit_keys
            .map(freq)
            .fillna(0)
            .astype(float)
        )

        X_apply[f'{prefix}_{col}_freq'] = (
            apply_keys
            .map(freq)
            .fillna(0)
            .astype(float)
        )

    return X_fit, X_apply


# ============================================================
# EXACT 33B DIGIT DECOMPOSITION
# ============================================================

def add_digit_features(X):
    X = X.copy()

    for col in numeric_cols:

        values = pd.to_numeric(
            X[col],
            errors='coerce'
        )

        integer_values = values.abs().round()

        X[f'{col}__digits'] = (
            np.floor(
                np.log10(
                    integer_values.clip(lower=1)
                )
            ) + 1
        )

        divisor = (
            10 ** (
                X[f'{col}__digits'] - 1
            )
        )

        X[f'{col}__first_digit'] = (
            integer_values / divisor
        ).fillna(0)

        X[f'{col}__first_digit'] = np.floor(
            X[f'{col}__first_digit']
        )

        X[f'{col}__last_digit'] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 10
        )

        def digit_sum(v):
            if pd.isna(v):
                return np.nan

            s = str(int(abs(v)))

            return sum(
                int(ch)
                for ch in s
            )

        X[f'{col}__digit_sum'] = (
            integer_values.map(digit_sum)
        )

        X[f'{col}__parity'] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 2
        )

        X[f'{col}__mod100'] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 100
        )

        X[f'{col}__mod1000'] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 1000
        )

        X[f'{col}__ends_zero'] = (
            integer_values
            .fillna(0)
            .astype(np.int64)
            % 10 == 0
        ).astype(np.int8)

    return X


# ============================================================
# 33B XGBOOST
# ============================================================

X_train_xgb = X_train_raw.copy()
X_test_xgb = X_test_raw.copy()

X_train_xgb, X_test_xgb = add_full_identity_features(
    X_train_xgb,
    X_test_xgb,
    y,
    numeric_cols,
    'base',
    smoothing=20
)

X_train_xgb = add_digit_features(X_train_xgb)
X_test_xgb = add_digit_features(X_test_xgb)

numeric_features = (
    X_train_xgb
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

categorical_features = [
    c for c in X_train_xgb.columns
    if c not in numeric_features
]

preprocessor = ColumnTransformer([
    (
        'num',
        SimpleImputer(strategy='median'),
        numeric_features
    ),
    (
        'cat',
        Pipeline([
            (
                'imputer',
                SimpleImputer(
                    strategy='most_frequent'
                )
            ),
            (
                'onehot',
                OneHotEncoder(
                    handle_unknown='ignore'
                )
            )
        ]),
        categorical_features
    )
])

print('Encoding XGBoost features...')

X_train_encoded = preprocessor.fit_transform(
    X_train_xgb
)

X_test_encoded = preprocessor.transform(
    X_test_xgb
)

print(
    'XGB encoded train shape:',
    X_train_encoded.shape
)

print(
    'XGB encoded test shape:',
    X_test_encoded.shape
)

xgb_model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

print('Training XGBoost...')

xgb_model.fit(
    X_train_encoded,
    y,
    verbose=False
)

xgb_pred = (
    xgb_model
    .predict_proba(X_test_encoded)[:, 1]
)

print('XGBoost predictions generated.')


# ============================================================
# EXP39 CATBOOST SEED 7
# ============================================================

X_train_cat = X_train_raw.copy()
X_test_cat = X_test_raw.copy()

sub_train = (
    X_train_cat['Subsidy_Available']
    .astype('string')
    .fillna('__MISSING__')
    .eq('Yes')
    .astype(int)
)

home_train = (
    X_train_cat['Home_Charging_Possible']
    .astype('string')
    .fillna('__MISSING__')
    .eq('Yes')
    .astype(int)
)

sub_test = (
    X_test_cat['Subsidy_Available']
    .astype('string')
    .fillna('__MISSING__')
    .eq('Yes')
    .astype(int)
)

home_test = (
    X_test_cat['Home_Charging_Possible']
    .astype('string')
    .fillna('__MISSING__')
    .eq('Yes')
    .astype(int)
)

X_train_cat['Subsidy_x_EnvConcern'] = (
    sub_train
    * X_train_cat['Environmental_Concern_Level']
)

X_train_cat['Subsidy_x_Income'] = (
    sub_train
    * X_train_cat['Annual_Income_USD']
)

X_train_cat['Subsidy_x_HomeCharging'] = (
    sub_train
    * home_train
)

X_test_cat['Subsidy_x_EnvConcern'] = (
    sub_test
    * X_test_cat['Environmental_Concern_Level']
)

X_test_cat['Subsidy_x_Income'] = (
    sub_test
    * X_test_cat['Annual_Income_USD']
)

X_test_cat['Subsidy_x_HomeCharging'] = (
    sub_test
    * home_test
)

value_identity_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work'
]

for col in value_identity_cols:

    X_train_cat[
        f'{col}__value_id'
    ] = (
        X_train_cat[col]
        .astype('string')
        .fillna('__MISSING__')
    )

    X_test_cat[
        f'{col}__value_id'
    ] = (
        X_test_cat[col]
        .astype('string')
        .fillna('__MISSING__')
    )

cat_identity_cols = (
    categorical_cols
    + [
        f'{c}__value_id'
        for c in value_identity_cols
    ]
)

for col in cat_identity_cols:

    X_train_cat[col] = (
        X_train_cat[col]
        .astype('string')
        .fillna('__MISSING__')
    )

    X_test_cat[col] = (
        X_test_cat[col]
        .astype('string')
        .fillna('__MISSING__')
    )

cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=3,
    random_strength=1,
    bootstrap_type='Bayesian',
    bagging_temperature=1,
    random_seed=7,
    thread_count=-1,
    verbose=False,
    allow_writing_files=False
)

print('Training CatBoost Seed 7...')

cat_model.fit(
    X_train_cat,
    y,
    cat_features=cat_identity_cols,
    verbose=False
)

cat_pred = (
    cat_model
    .predict_proba(X_test_cat)[:, 1]
)

print('CatBoost predictions generated.')


# ============================================================
# EXP41D WINNING RANK BLEND
# ============================================================

cat_rank = (
    pd.Series(cat_pred)
    .rank(method='average', pct=True)
    .to_numpy()
)

xgb_rank = (
    pd.Series(xgb_pred)
    .rank(method='average', pct=True)
    .to_numpy()
)

CAT_WEIGHT = 0.45
XGB_WEIGHT = 0.55

final_pred = (
    CAT_WEIGHT * cat_rank
    + XGB_WEIGHT * xgb_rank
)


# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({
    'id': test['id'],
    TARGET: final_pred
})

output_path = '../submissions/submission_13.csv'

submission.to_csv(
    output_path,
    index=False
)

print('')
print('=' * 70)
print('SUBMISSION 13 CREATED')
print('=' * 70)

print('Rows:', len(submission))
print('Columns:', list(submission.columns))
print('CatBoost rank weight:', CAT_WEIGHT)
print('XGBoost rank weight:', XGB_WEIGHT)
print('Prediction min:', final_pred.min())
print('Prediction max:', final_pred.max())
print('Prediction mean:', final_pred.mean())
print('Saved to:', output_path)

print('')
print(submission.head())

assert len(submission) == len(test)
assert list(submission.columns) == [
    'id',
    'Will_Buy_EV'
]
assert submission['Will_Buy_EV'].notna().all()
assert submission['Will_Buy_EV'].between(0, 1).all()
assert submission['id'].equals(test['id'])

print('')
print('ALL SUBMISSION CHECKS PASSED.')


Encoding XGBoost features...
XGB encoded train shape: (668665, 94)
XGB encoded test shape: (286571, 94)
Training XGBoost...
XGBoost predictions generated.
Training CatBoost Seed 7...
CatBoost predictions generated.

SUBMISSION 13 CREATED
Rows: 286571
Columns: ['id', 'Will_Buy_EV']
CatBoost rank weight: 0.45
XGBoost rank weight: 0.55
Prediction min: 6.0194506771445825e-05
Prediction max: 0.9999753987667978
Prediction mean: 0.5000017447683122
Saved to: ../submissions/submission_13.csv

       id  Will_Buy_EV
0  668665     0.567096
1  668666     0.476120
2  668667     0.290696
3  668668     0.281873
4  668669     0.595518

ALL SUBMISSION CHECKS PASSED.
